<a href="https://colab.research.google.com/github/mjjaiavinash/avinash-codebooster-2026/blob/main/day7_miniproject.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# =====================================================
# SMART NOTES SEARCH ENGINE
# Semantic Search using ChromaDB
# =====================================================

!pip install chromadb sentence-transformers

# =====================================================
# STEP 1: IMPORT LIBRARIES
# =====================================================

import pandas as pd
import chromadb

from sentence_transformers import SentenceTransformer

In [ ]:
# =====================================================
# STEP 2: LOAD DATASET
# =====================================================

notes_df = pd.read_csv(
    '/content/college_notes.csv'
)

print("===== Dataset Loaded =====")
print("Total Notes:", len(notes_df))
print("Columns:", notes_df.columns.tolist())

===== Dataset Loaded =====
Total Notes: 15
Columns: ['note_id', 'subject', 'topic', 'content']


In [ ]:
# =====================================================
# STEP 3: VIEW SAMPLE NOTE
# =====================================================

sample_note = notes_df.iloc[0]

print("\n===== Sample Note =====")

print("Note ID :", sample_note['note_id'])
print("Subject :", sample_note['subject'])
print("Topic   :", sample_note['topic'])
print("Content :", sample_note['content'])


===== Sample Note =====
Note ID : 1
Subject : Data Engineering
Topic   : ETL
Content : ETL is used to clean and transform raw data before analysis.


In [ ]:
# =====================================================
# STEP 4: PREPARE DOCUMENTS
# =====================================================

all_documents = notes_df['content'].tolist()

all_ids = notes_df['note_id'].astype(str).tolist()

all_metadata = [
    {
        "subject": row['subject'],
        "topic": row['topic']
    }
    for _, row in notes_df.iterrows()
]

print("\n===== Documents Prepared =====")

print("Documents:", len(all_documents))
print("IDs:", len(all_ids))
print("Metadata:", len(all_metadata))


===== Documents Prepared =====
Documents: 15
IDs: 15
Metadata: 15


In [ ]:
# =====================================================
# STEP 5: LOAD EMBEDDING MODEL
# =====================================================

print("\nLoading Embedding Model...")

model = SentenceTransformer(
    'all-MiniLM-L6-v2'
)

print("Model Loaded Successfully")



Loading Embedding Model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model Loaded Successfully


In [ ]:
# =====================================================
# STEP 6: GENERATE EMBEDDINGS
# =====================================================

embeddings = model.encode(all_documents)

print("\n===== Embeddings Generated =====")

print("Embedding Shape:", embeddings.shape)


===== Embeddings Generated =====
Embedding Shape: (15, 384)


In [ ]:
# =====================================================
# STEP 7: SETUP CHROMADB
# =====================================================

client = chromadb.Client()

# Delete the collection if it already exists to ensure a fresh start
try:
    client.delete_collection(name="college_notes")
    print("Existing collection 'college_notes' deleted.")
except: # noqa: E722
    print("Collection 'college_notes' did not exist or could not be deleted.")

collection = client.get_or_create_collection(
    name="college_notes"
)

print("\nChromaDB Collection Created")

Existing collection 'college_notes' deleted.

ChromaDB Collection Created


In [ ]:
# =====================================================
# STEP 8: INDEX ALL NOTES
# =====================================================

collection.add(
    documents=all_documents,
    embeddings=embeddings.tolist(),
    metadatas=all_metadata,
    ids=all_ids
)

print("\n===== Notes Indexed Successfully =====")


===== Notes Indexed Successfully =====


In [ ]:
# =====================================================
# STEP 9: SEMANTIC SEARCH QUERY
# =====================================================

query = "How does ETL clean data?"

print("\n===== User Query =====")
print(query)

# Generate query embedding

query_embedding = model.encode(query).tolist()

# Search similar notes

results = collection.query(
    query_embeddings=[query_embedding],
    n_results=3
)


===== User Query =====
How does ETL clean data?


In [ ]:
# =====================================================
# STEP 10: DISPLAY SEARCH RESULTS
# =====================================================

print("\n===== Semantic Search Results =====")

for i, doc in enumerate(results['documents'][0]):

    print(f"\nResult {i+1}")

    print("Document:", doc)

    print("Metadata:",
          results['metadatas'][0][i])


===== Semantic Search Results =====

Result 1
Document: ETL is used to clean and transform raw data before analysis.
Metadata: {'topic': 'ETL', 'subject': 'Data Engineering'}

Result 2
Document: Data pipelines automate extraction transformation and loading workflows.
Metadata: {'topic': 'Data Pipeline', 'subject': 'Data Engineering'}

Result 3
Document: Parquet is a columnar storage format optimized for analytics.
Metadata: {'topic': 'Parquet', 'subject': 'Data Engineering'}


In [ ]:
# =====================================================
# STEP 11: FILTER BY SUBJECT
# =====================================================

print("\n===== Subject Filter Search =====")

filtered_results = collection.query(
    query_embeddings=[query_embedding],
    n_results=2,

    where={
        "subject": "Data Engineering"
    }
)

for i, doc in enumerate(
        filtered_results['documents'][0]):

    print(f"\nFiltered Result {i+1}")

    print("Document:", doc)

    print(
        "Metadata:",
        filtered_results['metadatas'][0][i]
    )

# =====================================================
# END OF PROJECT
# =====================================================


===== Subject Filter Search =====

Filtered Result 1
Document: ETL is used to clean and transform raw data before analysis.
Metadata: {'subject': 'Data Engineering', 'topic': 'ETL'}

Filtered Result 2
Document: Data pipelines automate extraction transformation and loading workflows.
Metadata: {'topic': 'Data Pipeline', 'subject': 'Data Engineering'}
